# The Complete Guide on Neural Networks using PyTorch

Welcome to Deep Learning. A Neural Network is fundamentally a sequence of differentiable matrix multiplications. 

### The Mathematical Pipeline:
1. **The Forward Pass:** Input data ($X$) is multiplied by a Weight matrix ($W$), a Bias vector is added ($b$), and the result is passed through a non-linear Activation function ($f$).
   $$A = f(X \cdot W + b)$$
2. **The Loss Calculation:** We measure how far off the network's prediction ($\hat{y}$) is from the true target ($y$) using a Loss Function.
3. **Backpropagation:** Using the chain rule of calculus, we calculate the gradient (slope) of the Loss Function with respect to every single weight and bias in the network.
4. **Gradient Descent:** We update the weights in the opposite direction of the gradient to minimize the loss.

In this tutorial, we will build an analytical pipeline to classify a complex, non-linear dataset that standard machine learning algorithms would fail to solve.

In [ ]:
# Cell 2: Environment Setup and Imports
# We import PyTorch (torch) and its neural network module (nn).
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# Set seeds for analytical reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch Version: {torch.__version__}")

## Step 1: Generating a Non-Linear Dataset

To prove the power of Neural Networks, we need data that cannot be separated by a simple straight line. We will use the `make_moons` dataset, which creates two interlocking half-circles. 

Because neural networks require tensors to perform GPU-accelerated matrix math, we must convert our standard NumPy arrays into `torch.Tensor` objects.

In [ ]:
# Cell 4: Data Generation and Tensor Conversion

# 1. Generate 1000 samples of noisy interlocking moons
X, y = make_moons(n_samples=1000, noise=0.15, random_state=42)

# 2. Split into Training (80%) and Testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Convert NumPy arrays to PyTorch Tensors
# X features must be FloatTensors for matrix multiplication
X_train_tensor = torch.FloatTensor(X_train)
X_test_tensor = torch.FloatTensor(X_test)

# y labels must be LongTensors (integers) for classification loss functions
y_train_tensor = torch.LongTensor(y_train)
y_test_tensor = torch.LongTensor(y_test)

# Visualize the dataset
plt.figure(figsize=(8, 6))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='coolwarm', edgecolors='k', alpha=0.8)
plt.title("The 'Moons' Dataset: A Non-Linear Challenge", fontsize=14)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

## Step 2: Defining the Neural Network Architecture

In PyTorch, we build models by creating a class that inherits from `nn.Module`. 

We will build a Multi-Layer Perceptron (MLP) with one hidden layer:
* **Input Layer:** 2 neurons (because we have 2 features: X and Y coordinates).
* **Hidden Layer:** 16 neurons. We apply a **ReLU** (Rectified Linear Unit) activation function here. ReLU simply outputs $x$ if $x > 0$, and $0$ if $x \le 0$. This mathematically introduces the non-linearity needed to bend the decision boundary.
* **Output Layer:** 2 neurons (because we are classifying into 2 categories: Red Moon or Blue Moon).

In [ ]:
# Cell 6: Building the PyTorch Model

class MoonClassifier(nn.Module):
    def __init__(self):
        super(MoonClassifier, self).__init__()
        
        # Define the mathematical layers
        # nn.Linear(input_features, output_features) performs: y = xA^T + b
        self.hidden_layer = nn.Linear(in_features=2, out_features=16)
        self.relu = nn.ReLU() # The Activation Function
        self.output_layer = nn.Linear(in_features=16, out_features=2)
        
    def forward(self, x):
        """
        The forward method defines how data flows through the network.
        PyTorch uses this to automatically build the computational graph for Backpropagation.
        """
        # Step 1: Pass inputs through the hidden layer
        out = self.hidden_layer(x)
        
        # Step 2: Apply non-linear activation
        out = self.relu(out)
        
        # Step 3: Pass to the output layer to get raw logits
        logits = self.output_layer(out)
        
        return logits

# Instantiate the model
model = MoonClassifier()
print(model)

## Step 3: The Loss Function and Optimizer

To train the network, we need two components:
1. **The Criterion (Loss Function):** We use `CrossEntropyLoss`. This mathematically combines a Softmax activation (to turn raw logits into probabilities) and Negative Log Likelihood (to measure the error).
2. **The Optimizer:** We use **Adam** (Adaptive Moment Estimation). It is an advanced version of Gradient Descent that maintains a per-parameter learning rate, allowing for faster and more stable convergence.

In [ ]:
# Cell 8: Instantiating Loss and Optimizer

# Define the Loss Function
criterion = nn.CrossEntropyLoss()

# Define the Optimizer
# We pass model.parameters() so the optimizer knows WHICH weights it is allowed to update.
# lr=0.01 is the Learning Rate (the step size taken during gradient descent).
optimizer = optim.Adam(model.parameters(), lr=0.01)

print("Loss function and Optimizer configured.")

## Step 4: The Training Loop

This is the beating heart of Machine Learning. Unlike standard Python scripts, PyTorch requires you to explicitly write the training loop. 

**For every epoch (pass over the data), we follow 5 strict mathematical steps:**
1. **Forward Pass:** Pass data through the model to get predictions.
2. **Compute Loss:** Calculate the error against the true labels.
3. **Zero Gradients:** Clear old gradients from the last step (PyTorch accumulates them by default).
4. **Backward Pass:** Calculate the derivative of the loss w.r.t every weight (`loss.backward()`).
5. **Optimizer Step:** Update the weights (`optimizer.step()`).

In [ ]:
# Cell 10: Executing the Training Loop

epochs = 1000
loss_history = []

print("--- Starting Training ---")
for epoch in range(epochs):
    # 1. Forward Pass
    y_pred_logits = model(X_train_tensor)
    
    # 2. Compute Loss
    loss = criterion(y_pred_logits, y_train_tensor)
    loss_history.append(loss.item())
    
    # 3. Zero Gradients
    optimizer.zero_grad()
    
    # 4. Backward Pass (Calculates the gradients)
    loss.backward()
    
    # 5. Optimizer Step (Updates the weights)
    optimizer.step()
    
    # Analytical logging
    if (epoch + 1) % 200 == 0:
        # Calculate accuracy for monitoring
        _, predicted_classes = torch.max(y_pred_logits, 1)
        correct = (predicted_classes == y_train_tensor).sum().item()
        accuracy = correct / len(y_train_tensor)
        
        print(f"Epoch {epoch+1:4d} | Loss: {loss.item():.4f} | Training Accuracy: {accuracy*100:.1f}%")

print("--- Training Complete ---")

## Step 5: Analytical Evaluation and Visualization

How did our network learn to separate the interlocking moons? By combining 16 different linear boundaries (from the hidden neurons) and bending them using the ReLU activation function, it creates a complex, continuous decision boundary.

Let's evaluate the model on our unseen Test Set and plot the mathematical boundary it learned across the entire 2D space.

In [ ]:
# Cell 12: Evaluation and Decision Boundary Visualization

# 1. Evaluate on Test Set
# Use torch.no_grad() to save memory, as we don't need to calculate gradients for evaluation
with torch.no_grad():
    test_logits = model(X_test_tensor)
    _, test_predictions = torch.max(test_logits, 1)
    
    correct = (test_predictions == y_test_tensor).sum().item()
    test_accuracy = correct / len(y_test_tensor)
    print(f"Final Test Accuracy: {test_accuracy*100:.2f}%")

# 2. Plotting the Decision Boundary
# Create a grid of points across the feature space
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

# Flatten the grid and pass it through the model
grid_tensor = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])
with torch.no_grad():
    Z_logits = model(grid_tensor)
    _, Z_preds = torch.max(Z_logits, 1)

# Reshape predictions back to the grid shape
Z = Z_preds.numpy().reshape(xx.shape)

# Plotting
plt.figure(figsize=(10, 7))
# Plot the decision regions
plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
# Plot the actual test data points
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='coolwarm', edgecolors='k', s=50)

plt.title(f"Neural Network Decision Boundary (Test Accuracy: {test_accuracy*100:.1f}%)", fontsize=14)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

print("\nAnalytical Conclusion: The neural network successfully warped its internal representation to draw an 'S' shape between the two classes. Standard linear models would have failed entirely here.")